# Learning a PC with Expectation Maximization

In this notebook we will show how to train a circuit using EM. Most of the logic will be reused from [learning a circuit](./learning-a-circuit.ipynb), please look at this notebook for more informations.

In [24]:
%load_ext autoreload
%autoreload 2
from cirkit.templates import data_modalities, utils
from cirkit.pipeline import PipelineContext
import random
import numpy as np
import torch


def get_circuit():
    """Function used to get a fresh, untrained, circuit in the tutorial"""
    symbolic_circuit = data_modalities.image_data(
        (1, 28, 28),
        region_graph="quad-graph",
        input_layer="categorical",
        num_input_units=64,
        sum_product_layer="cp",
        num_sum_units=64,
        sum_weight_param=utils.Parameterization(
            activation="none",  # This time, we do not use a softmax function as EM will keep the parameters convex
            initialization="uniform",
            initialization_kwargs={"convex": True},
        ),
        input_params={
            "probs": utils.Parameterization(
                activation="none",  # This time, we do not use a softmax function as EM will keep the parameters convex
                initialization="uniform",
                initialization_kwargs={"convex": True},
            )
        },
    )
    ctx = PipelineContext(backend="torch", optimize=True, fold=True, semiring="lse-sum")
    circuit = ctx.compile(symbolic_circuit)
    return circuit


# Set some seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Set the torch device to use
device = torch.device("cuda")
circuit = get_circuit()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


TypeError: GaussianLayer.__init__() got an unexpected keyword argument 'probs_factory'

We can check that our weight are indeed convex after initialization.

In [13]:
def check_convex(cc):
    for layer in cc.layers:
        param = list(layer.parameters())
        if len(param) == 0:
            continue
        if not ((param[0].sum(dim=-1) - 1.0) < 1e-5).all():
            return False
    return True


check_convex(circuit)

True

We also want to enable EM on our circuit, for now we will enable it on every layer manually

In [14]:
def enable_em(cc):
    for layer in cc.layers:
        layer.enable_em()


enable_em(circuit)

## Manual EM

### Mini Batch Stochastic EM

Everything is ready to start training. Here we will use the `em_accumulate` and `em_step` to, respectively, compute the em update and apply it. This separation allows us to use batch accumulation but for now, let's do a simple mini batch approach. 

In [15]:
from torch.utils.data import DataLoader
from torchvision import transforms, datasets

# Load the MNIST data set and data loaders
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        # Flatten the images and set pixel values in the [0-255] range
        transforms.Lambda(lambda x: (255 * x.view(-1)).long()),
    ]
)
data_train = datasets.MNIST("datasets", train=True, download=True, transform=transform)
data_test = datasets.MNIST("datasets", train=False, download=True, transform=transform)

# Instantiate the training and testing data loaders
train_dataloader = DataLoader(data_train, shuffle=True, batch_size=256)
test_dataloader = DataLoader(data_test, shuffle=False, batch_size=256)

In [16]:
num_epochs = 10
step_idx = 0
running_loss = 0.0
running_samples = 0

# Move the circuit to chosen device
circuit = circuit.to(device)


def accumulate_layers(cc):
    for l in cc.layers:
        l.em_accumulate()


def step_layers(cc):
    for l in cc.layers:
        l.em_step(0.2, 1e-8)


for epoch_idx in range(num_epochs):
    for i, (batch, _) in enumerate(train_dataloader):
        # The circuit expects an input of shape (batch_dim, num_variables)
        batch = batch.to(device)

        # Compute the log-likelihoods of the batch, by evaluating the circuit
        log_likelihoods = circuit(batch)

        # We take the average log-likelihood as loss (Note that we are not using the negated value!)
        loss = torch.mean(log_likelihoods)
        loss.backward()
        # First accumulate the em update, then perform a step
        accumulate_layers(circuit)
        step_layers(circuit)

        # Zero grad the layers
        circuit.zero_grad()

        running_loss += loss.detach() * len(batch)
        running_samples += len(batch)
        step_idx += 1
    average_nll = running_loss / running_samples
    print(f"Step {step_idx}: Average NLL: {average_nll:.3f}")
    running_loss = 0.0
    running_samples = 0

Step 235: Average NLL: -956.843
Step 470: Average NLL: -815.456
Step 705: Average NLL: -760.038
Step 940: Average NLL: -743.184
Step 1175: Average NLL: -735.148
Step 1410: Average NLL: -730.880
Step 1645: Average NLL: -727.848
Step 1880: Average NLL: -725.035
Step 2115: Average NLL: -722.341
Step 2350: Average NLL: -720.555


### Full EM

Now, we will use the batch accumulation do to a full batch EM.

In [17]:
train_dataloader = DataLoader(data_train, shuffle=True, batch_size=1000)
test_dataloader = DataLoader(data_test, shuffle=False, batch_size=1000)

In [18]:
num_epochs = 10
step_idx = 0
running_loss = 0.0
running_samples = 0

# Move the circuit to chosen device
circuit = get_circuit().to(device)
enable_em(circuit)


def accumulate_layers(cc):
    for l in cc.layers:
        l.em_accumulate()


def step_layers(cc):
    for l in cc.layers:
        l.em_step(1, 1e-8)


for epoch_idx in range(num_epochs):
    for i, (batch, _) in enumerate(train_dataloader):
        # The circuit expects an input of shape (batch_dim, num_variables)
        batch = batch.to(device)

        # Compute the log-likelihoods of the batch, by evaluating the circuit
        log_likelihoods = circuit(batch)

        # We take the average log-likelihood as loss (Note that we are not using the negated value!)
        loss = torch.mean(log_likelihoods)
        loss.backward()

        # Accumulate the em update without updating the weight
        accumulate_layers(circuit)

        running_loss += loss.detach() * len(batch)
        running_samples += len(batch)
        step_idx += 1

    step_layers(circuit)
    circuit.zero_grad()
    average_nll = running_loss / running_samples
    print(f"Step {step_idx}: Average NLL: {average_nll:.3f}")
    running_loss = 0.0
    running_samples = 0

Step 60: Average NLL: -4351.470
Step 120: Average NLL: -917.782
Step 180: Average NLL: -917.774
Step 240: Average NLL: -917.766
Step 300: Average NLL: -917.756
Step 360: Average NLL: -917.746
Step 420: Average NLL: -917.734
Step 480: Average NLL: -917.720
Step 540: Average NLL: -917.704
Step 600: Average NLL: -917.684


## Using the Optimizer to simplify the process
We provide a pytorch-like optimizer that automatically handle the process and allows you to use it as a drop-in replacement for Adam.

### Stochastic EM with optimizer

In [25]:
from cirkit.backend.torch.em_optimizer import EM

train_dataloader = DataLoader(data_train, shuffle=True, batch_size=256)
test_dataloader = DataLoader(data_test, shuffle=False, batch_size=256)

circuit = get_circuit()
optim = EM(circuit, 0.2, 1e-8)

num_epochs = 10
step_idx = 0
running_loss = 0.0
running_samples = 0

# Move the circuit to chosen device
circuit = circuit.to(device)


for epoch_idx in range(num_epochs):
    for i, (batch, _) in enumerate(train_dataloader):
        # The circuit expects an input of shape (batch_dim, num_variables)
        batch = batch.to(device)

        # Compute the log-likelihoods of the batch, by evaluating the circuit
        log_likelihoods = circuit(batch)

        # We take the average log-likelihood as loss (Note that we are not using the negated value!)
        loss = torch.mean(log_likelihoods)
        loss.backward()
        optim.step()
        optim.zero_grad()
        running_loss += loss.detach() * len(batch)
        running_samples += len(batch)
        step_idx += 1

    average_nll = running_loss / running_samples
    print(f"Step {step_idx}: Average NLL: {average_nll:.3f}")
    running_loss = 0.0
    running_samples = 0

[autoreload of cirkit.backend.torch.utils failed: Traceback (most recent call last):
  File "/disk/scratch_fast1/s2893001/cirkit/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/disk/scratch_fast1/s2893001/cirkit/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/disk/scratch_fast1/s2893001/cirkit/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/disk/scratch_fast1/s2893001/cirkit/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 349, in update_class
    if update_generic(old_obj, new_obj):
  File "/disk/scratch_fast1/s2893001/cirkit/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/disk/scratch_fast1/s2893001/cirkit/.venv/lib/python3.

TypeError: GaussianLayer.__init__() got an unexpected keyword argument 'probs_factory'

### Full EM with optimizer

In [20]:
from cirkit.backend.torch.em_optimizer import EM

circuit = get_circuit()
optim = EM(circuit, 1, 1e-8)

num_epochs = 10
step_idx = 0
running_loss = 0.0
running_samples = 0

# Move the circuit to chosen device
circuit = circuit.to(device)


for epoch_idx in range(num_epochs):
    for i, (batch, _) in enumerate(train_dataloader):
        # The circuit expects an input of shape (batch_dim, num_variables)
        batch = batch.to(device)

        # Compute the log-likelihoods of the batch, by evaluating the circuit
        log_likelihoods = circuit(batch)

        # We take the average log-likelihood as loss (Note that we are not using the negated value!)
        loss = torch.mean(log_likelihoods)
        loss.backward()
        running_loss += loss.detach() * len(batch)
        running_samples += len(batch)
        step_idx += 1

    optim.step()
    optim.zero_grad()
    average_nll = running_loss / running_samples
    print(f"Step {step_idx}: Average NLL: {average_nll:.3f}")
    running_loss = 0.0
    running_samples = 0

Step 60: Average NLL: -4350.689
Step 120: Average NLL: -917.782
Step 180: Average NLL: -917.774
Step 240: Average NLL: -917.766
Step 300: Average NLL: -917.756
Step 360: Average NLL: -917.746
Step 420: Average NLL: -917.735
Step 480: Average NLL: -917.722
Step 540: Average NLL: -917.706
Step 600: Average NLL: -917.686


In [22]:
circuit

[autoreload of cirkit.backend.torch.utils failed: Traceback (most recent call last):
  File "/disk/scratch_fast1/s2893001/cirkit/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/disk/scratch_fast1/s2893001/cirkit/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/disk/scratch_fast1/s2893001/cirkit/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/disk/scratch_fast1/s2893001/cirkit/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 349, in update_class
    if update_generic(old_obj, new_obj):
  File "/disk/scratch_fast1/s2893001/cirkit/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/disk/scratch_fast1/s2893001/cirkit/.venv/lib/python3.

TorchCircuit(
  (0): TorchCategoricalLayer(
    folds: 784  variables: 1  output-units: 64
    input-shape: (784, 1, -1, 1)
    output-shape: (784, -1, 64)
    (probs): TorchParameter(
      shape: (784, 64, 256)
      (0): TorchTensorParameter(output-shape: (784, 64, 256))
    )
  )
  (1): TorchSumLayer(
    folds: 1568  arity: 1  input-units: 64  output-units: 64
    input-shape: (1568, 1, -1, 64)
    output-shape: (1568, -1, 64)
    (weight): TorchParameter(
      shape: (1568, 64, 64)
      (0): TorchTensorParameter(output-shape: (1568, 64, 64))
    )
  )
  (2): TorchCPTLayer(
    folds: 784  arity: 2  input-units: 64  output-units: 64
    input-shape: (784, 2, -1, 64)
    output-shape: (784, -1, 64)
    (weight): TorchParameter(
      shape: (784, 64, 64)
      (0): TorchTensorParameter(output-shape: (784, 64, 64))
    )
  )
  (3): TorchHadamardLayer(
    folds: 392  arity: 2  input-units: 64  output-units: 64
    input-shape: (392, 2, -1, 64)
    output-shape: (392, -1, 64)
  )
 